In [1]:
# ============================================================
# VEHICLE CLASS-WISE SPEED DATA CLEANING
# OUTLIER REMOVAL + NEW V85 CALCULATION
# FOR: Heavy Vehicle, Medium Light Vehicle, Two Wheelers
# ============================================================

import pandas as pd
import numpy as np
import os
from pathlib import Path

In [2]:
# ============================================================
# STEP 1: Define input and output folders
# ============================================================

input_folder = r"D:\MSC THESIS AAVAS\Vehicle class wise\initial data\BP"

output_folder = r"D:\MSC THESIS AAVAS\Vehicle class wise\Cleaned data\BP"

# Create output folder if it does not exist
os.makedirs(output_folder, exist_ok=True)

print("Input folder:", input_folder)
print("Output folder:", output_folder)

Input folder: D:\MSC THESIS AAVAS\Vehicle class wise\initial data\BP
Output folder: D:\MSC THESIS AAVAS\Vehicle class wise\Cleaned data\BP


In [3]:
# ============================================================
# STEP 2: Function to clean each sheet
# ============================================================

def clean_speed_data(df):
    """
    For each sheet:
    1. Fixes column names by removing extra spaces
    2. Removes outliers using IQR method
    3. Keeps only accepted speed values
    4. Calculates new 85th percentile speed from accepted values only
    5. Returns only:
       Chainage | Speed | Unit | New_85th_Percentile
    """

    # Remove extra spaces from column names
    df.columns = df.columns.astype(str).str.strip()

    # Check required columns
    if "Chainage" not in df.columns:
        raise ValueError("Chainage column not found")

    if "Speed" not in df.columns:
        raise ValueError("Speed column not found")

    # Convert speed to numeric
    df["Speed"] = pd.to_numeric(df["Speed"], errors="coerce")

    # Remove blank speed values
    df = df.dropna(subset=["Speed"]).copy()

    # Calculate Q1, Q3 and IQR
    Q1 = df["Speed"].quantile(0.25)
    Q3 = df["Speed"].quantile(0.75)
    IQR = Q3 - Q1

    # Outlier limits
    lower_limit = Q1 - 1.5 * IQR
    upper_limit = Q3 + 1.5 * IQR

    # Keep only accepted speeds
    cleaned_df = df[
        (df["Speed"] >= lower_limit) &
        (df["Speed"] <= upper_limit)
    ].copy()

    # Calculate new V85 from accepted speed only
    if len(cleaned_df) > 0:
        new_v85 = np.percentile(cleaned_df["Speed"], 85)
    else:
        new_v85 = np.nan

    # Add Unit column
    cleaned_df["Unit"] = "km/hr"

    # Add New V85 column
    cleaned_df["New_85th_Percentile"] = round(new_v85, 2)

    # Keep only required columns
    cleaned_df = cleaned_df[
        ["Chainage", "Speed", "Unit", "New_85th_Percentile"]
    ]

    # Summary information
    summary = {
        "Original_Count": len(df),
        "Cleaned_Count": len(cleaned_df),
        "Outliers_Removed": len(df) - len(cleaned_df),
        "Q1": Q1,
        "Q3": Q3,
        "IQR": IQR,
        "Lower_Limit": lower_limit,
        "Upper_Limit": upper_limit,
        "New_85th_Percentile": new_v85
    }

    return cleaned_df, summary

In [4]:
# ============================================================
# STEP 3: Process all Excel files and all sheets
# ============================================================

summary_list = []

# Read all Excel files from input folder
excel_files = list(Path(input_folder).glob("*.xlsx"))

# Avoid temporary Excel files
excel_files = [
    file for file in excel_files
    if not file.name.startswith("~$")
]

print(f"Total Excel files found: {len(excel_files)}")

for file_path in excel_files:

    file_name = file_path.name
    print(f"\nProcessing file: {file_name}")

    # Read all sheets from current file
    all_sheets = pd.read_excel(file_path, sheet_name=None)

    cleaned_sheets = {}

    for sheet_name, df in all_sheets.items():

        print(f"  Processing sheet: {sheet_name}")

        try:
            cleaned_df, summary = clean_speed_data(df)

            cleaned_sheets[sheet_name] = cleaned_df

            summary_list.append({
                "File_Name": file_name,
                "Sheet_Name": sheet_name,
                "Original_Count": summary["Original_Count"],
                "Cleaned_Count": summary["Cleaned_Count"],
                "Outliers_Removed": summary["Outliers_Removed"],
                "Q1": round(summary["Q1"], 2),
                "Q3": round(summary["Q3"], 2),
                "IQR": round(summary["IQR"], 2),
                "Lower_Limit": round(summary["Lower_Limit"], 2),
                "Upper_Limit": round(summary["Upper_Limit"], 2),
                "New_85th_Percentile": round(summary["New_85th_Percentile"], 2)
            })

        except Exception as e:
            print(f"    Skipped sheet '{sheet_name}' because: {e}")

    # Save cleaned workbook
    cleaned_output_path = os.path.join(
        output_folder,
        "Cleaned_" + file_name
    )

    if cleaned_sheets:
        with pd.ExcelWriter(cleaned_output_path, engine="openpyxl") as writer:
            for sheet_name, data in cleaned_sheets.items():
                data.to_excel(writer, sheet_name=sheet_name[:31], index=False)

        print(f"  Saved: {cleaned_output_path}")

print("\nAll Excel files cleaned successfully.")

Total Excel files found: 3

Processing file: Heavy Vehicle.xlsx
  Processing sheet: Sheet1
  Processing sheet: Sheet2
  Processing sheet: Sheet3
  Processing sheet: Sheet4
  Processing sheet: Sheet5
  Processing sheet: Sheet6
  Processing sheet: Sheet7
  Processing sheet: Sheet8
  Processing sheet: Sheet9
  Processing sheet: Sheet10
  Processing sheet: Sheet11
  Processing sheet: Sheet12
  Processing sheet: Sheet13
  Processing sheet: Sheet14
  Processing sheet: Sheet15
  Processing sheet: Sheet16
  Processing sheet: Sheet17
  Processing sheet: Sheet18
  Processing sheet: Sheet19
  Processing sheet: Sheet20
  Processing sheet: Sheet21
  Processing sheet: Sheet22
  Processing sheet: Sheet23
  Processing sheet: Sheet24
  Processing sheet: Sheet25
  Processing sheet: Sheet26
  Processing sheet: Sheet27
  Processing sheet: Sheet28
  Processing sheet: Sheet29
  Processing sheet: Sheet30
  Processing sheet: Sheet31
  Processing sheet: Sheet32
  Processing sheet: Sheet33
  Processing sheet: S

In [5]:
# ============================================================
# STEP 4: Save final V85 summary table
# ============================================================

summary_df = pd.DataFrame(summary_list)

summary_output_path = os.path.join(
    output_folder,
    "Final_V85_Summary_After_Outlier_Removal.xlsx"
)

summary_df.to_excel(summary_output_path, index=False)

print("\nFinal summary saved at:")
print(summary_output_path)

summary_df


Final summary saved at:
D:\MSC THESIS AAVAS\Vehicle class wise\Cleaned data\BP\Final_V85_Summary_After_Outlier_Removal.xlsx


,File_Name,Sheet_Name,Original_Count,Cleaned_Count,Outliers_Removed,Q1,Q3,IQR,Lower_Limit,Upper_Limit,New_85th_Percentile
0,Heavy Vehicle.xlsx,Sheet1,6,6,0,40.25,46.25,6.00,31.25,55.25,48.00
1,Heavy Vehicle.xlsx,Sheet2,9,9,0,25.00,28.00,3.00,20.50,32.50,28.80
2,Heavy Vehicle.xlsx,Sheet3,16,16,0,28.75,43.00,14.25,7.38,64.38,44.50
3,Heavy Vehicle.xlsx,Sheet4,9,9,0,29.00,32.00,3.00,24.50,36.50,32.80
4,Heavy Vehicle.xlsx,Sheet5,3,3,0,25.00,26.50,1.50,22.75,28.75,26.70
...,...,...,...,...,...,...,...,...,...,...,...
154,Two Wheelers.xlsx,Sheet49,43,42,1,39.50,48.00,8.50,26.75,60.75,52.85
155,Two Wheelers.xlsx,Sheet50,41,41,0,36.00,45.00,9.00,22.50,58.50,47.00
156,Two Wheelers.xlsx,Sheet51,44,42,2,33.50,43.00,9.50,19.25,57.25,44.00
157,Two Wheelers.xlsx,Sheet52,41,39,2,37.00,42.00,5.00,29.50,49.50,43.00
